# Task
Build an AI agent that assists students in exploring colleges based on their career interests.

## Agent Setup and Initial Interaction

### Subtask:
Set up the Groq agent and develop the initial conversational flow to understand the student's specific career interest and introduce the college exploration phase.


In [1]:
%pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 11.5 MB/s eta 0:00:00


In [2]:
!pip install -qU pinecone pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.0 MB/s eta 0:00:00


In [3]:
from getpass import getpass
import os

pinecone_api_key = getpass('Enter your Pinecone API Key: ')
os.environ["PINECONE_API_KEY"] = pinecone_api_key

Enter your Pinecone API Key: ··········


In [4]:
from google.adk.agents.llm_agent import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts
from google.genai.types import GenerateContentConfig
import os
import asyncio
import random

In [5]:
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Scraping College Websites

In [6]:
!pip install beautifulsoup4 requests # python package for parsing html and xml documents

import requests
from bs4 import BeautifulSoup

def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    paragraphs = [p.get_text() for p in soup.find_all("p")]
    return " ".join(paragraphs)

stanford_cs_text = scrape_page("https://cs.stanford.edu/")

In [7]:
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = chunk_text(stanford_cs_text)

# Create AI Agent

In [8]:
# Define the initial system message for the AI agent
college_planning_agent = Agent(
    name="college_planning_agent",
    model=LiteLlm(model="groq/llama-3.3-70b-versatile"), # Use LiteLlm with the specified model
    instruction="""
You are an agent designed to assist students in exploring colleges based on their career interests.

Given the user's career interest as provided after the original response in the conversational loop, please direct the conversation towards college planning
and helping the user explore colleges based on requirements and career goals.
Keep answers concise and exploratory. Answers should mimic a college advisor tone and focus on the subject of colleges and college planning.
"""
)

session_service = InMemorySessionService()

  # Define constants for identifying the interaction context
APP_NAME = "college_workflow"
USER_ID = "user_1"
SESSION_ID = "session_001" # Using a fixed ID for simplicity

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

runner = Runner(
    app_name=APP_NAME,
    agent=college_planning_agent,
    session_service=session_service
)

async def call_agent_async(query: str, runner, user_id, session_id):
    """
    Sends a query to the agent, prints responses, and allows the user to
    continue replying if the agent asks follow-up questions.
    """

    user_message = query
    turn = 1

    while True:
        print(f"\n>>> User ({turn}): {user_message}")

        # Convert user message to ADK content format
        content = types.Content(
            role='user',
            parts=[types.Part(text=user_message)]
        )

        final_response_text = None

        # Run the agent and process events
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=content
        ):
            # Debug (optional)
            # print(f"[Event] Author={event.author}, Final={event.is_final_response()}, Content={event.content}")

            if event.is_final_response():
                if event.content and event.content.parts:
                    final_response_text = event.content.parts[0].text
                break

        # Display agent message
        print(f"<<< Agent: {final_response_text}")

        # Determine whether the agent expects more user input
        # Usually when the final response ends with a question or asks for info.
        if final_response_text and final_response_text.strip().endswith("?"):
            # Agent is asking a follow-up question → get user reply
            user_message = input("You: ")
            turn += 1
            continue

        # Stop if no more questions
        break

In [9]:
async def run_conversation(query):
    await call_agent_async(query,runner=runner,user_id=USER_ID, session_id=SESSION_ID)

In [ ]:
# Write a query to test your workflow
query = "Start a college advising session with the student."
await run_conversation(query)


>>> User (1): Start a college advising session with the student.
<<< Agent: Welcome to our college planning session. I'm excited to help you explore colleges that align with your career goals. To get started, can you tell me a bit about your career interests? What field or industry are you considering for your future career?


# Task
Identify California universities and their URLs for academic programs, majors, and course catalogs, potentially using educational directories or a predefined list.

## Identify California Universities and URLs

### Subtask:
Gather a comprehensive list of California universities and their corresponding URLs for academic programs, majors, and course catalogs.


```markdown
### Strategy for Gathering California University Information

To accomplish this subtask, we will follow a hybrid approach:

1.  **Initial University List Acquisition (Programmatic):** We will programmatically scrape a reputable source (e.g., Wikipedia or an official education directory) to obtain an initial list of California university names and their main website URLs. This provides a structured starting point.

2.  **Targeted URL Discovery (Programmatic with Heuristics):** For each university obtained in step 1, we will develop a function that attempts to locate specific URLs for "Academic Programs", "Majors", and "Course Catalogs". This will involve:
    *   Making requests to the university's main website.
    *   Using `BeautifulSoup` to parse the HTML.
    *   Searching for common keywords (e.g., "Academics", "Programs", "Majors", "Courses", "Catalog") in links and text content.
    *   Employing regular expressions or pattern matching to identify potential URLs based on common structures.

3.  **Data Structuring:** The collected information will be stored in a dictionary, with university names as keys and nested dictionaries containing the found URLs.

**Note:** Web scraping can be complex and sometimes unreliable due to varying website structures. Our programmatic approach will aim to be robust but might require manual verification or refinement for certain university websites.
```

### Step 1: Programmatically Acquire Initial List of California Universities

We will use Wikipedia's 'List of colleges and universities in California' page as our initial data source. This page provides a structured list of institutions, which can be easily parsed to extract university names and their main website URLs. We'll use `requests` to fetch the page content and `BeautifulSoup` to parse the HTML and extract the relevant information from tables.

**Reasoning**:
Now that the strategy for acquiring the initial list of California universities from Wikipedia has been outlined, the next step is to implement the scraping process using `requests` and `BeautifulSoup` to extract university names and their primary website URLs.



In [ ]:
import requests
from bs4 import BeautifulSoup

wikipedia_url = "https://en.wikipedia.org/wiki/List_of_colleges_and_universities_in_California"

# Add a User-Agent header to the request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(wikipedia_url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

california_universities = {}

# Focus on 'wikitable sortable' tables, as they contain structured lists of universities
sortable_tables = soup.find_all('table', class_='wikitable sortable')

print(f"Found {len(sortable_tables)} tables with class 'wikitable sortable'.\n")

for table_idx, table in enumerate(sortable_tables):
    rows = table.find_all('tr')

    # Skip if no rows or only header row
    if not rows or len(rows) < 2:
        print(f"Skipping table {table_idx} (wikitable sortable): not enough rows.")
        continue

    # Extract headers from the first row to determine column indices
    headers_raw = [th.get_text(strip=True) for th in rows[0].find_all('th')]
    headers_lower = [h.lower() for h in headers_raw]
    print(f"Processing Table {table_idx} (wikitable sortable) headers: {headers_raw}")

    university_name_col_index = -1

    # Find the column index for the university name
    for idx, header in enumerate(headers_lower):
        if "institution" in header or "name" in header or "university" in header or "school" in header:
            university_name_col_index = idx
            break

    # If no university name column is found, skip this table
    if university_name_col_index == -1:
        print(f"Skipping table {table_idx} (wikitable sortable): No suitable university name column found.")
        continue

    # Iterate over data rows (skip header row)
    for row_idx, row in enumerate(rows[1:]):
        # Limit debug output for rows
        # if row_idx >= 5:
        #     break

        cols = row.find_all('td') # Data rows should only contain 'td' elements

        # Ensure there are enough columns to access the university name index
        if len(cols) > university_name_col_index:
            university_name_tag = cols[university_name_col_index].find('a')

            if university_name_tag and university_name_tag.text:
                university_name = university_name_tag.text.strip()

                main_url = None
                # Search for the first valid external link within any column of the current row
                for col in cols:
                    website_links = col.find_all('a', href=True)
                    for link in website_links:
                        href = link['href']
                        # Check if the link is an external website and not a Wikipedia page
                        if href.startswith(('http://', 'https://')) and 'wikipedia.org' not in href:
                            main_url = href
                            break
                    if main_url: # Stop searching if a URL is found in this row
                        break

                if main_url:
                    california_universities[university_name] = {
                        'main_url': main_url,
                        'academic_programs_url': None,
                        'majors_url': None,
                        'course_catalog_url': None
                    }
                    # print(f"    Extracted: {university_name} - {main_url}") # Too verbose
                # else:
                    # print(f"    No valid main_url found for {university_name} in row {row_idx}. Cols: {[c.get_text(strip=True) for c in cols]}") # Debugging
            # else:
                # print(f"    No university name tag found in row {row_idx} at col {university_name_col_index}. Cols: {[c.get_text(strip=True) for c in cols]}") # Debugging
        # else:
            # print(f"    Not enough columns in row {row_idx}. Cols length: {len(cols)}") # Debugging

print(f"\nFinal count: Found {len(california_universities)} universities.")
print("First 5 extracted universities:")
for i, (name, data) in enumerate(california_universities.items()):
    if i >= 5: break
    print(f"- {name}: {data['main_url']}")

Found 5 tables with class 'wikitable sortable'.

Processing Table 0 (wikitable sortable) headers: ['Name', 'City', 'County', 'Enrollment[1](Fall 2024)', 'Founded', 'Athletics']
Processing Table 1 (wikitable sortable) headers: ['Name', 'City', 'County', 'Enrollment[1](Fall 2024)', 'Founded', 'Athletics']
Processing Table 2 (wikitable sortable) headers: ['Name', 'City', 'County', 'Founded']
Processing Table 3 (wikitable sortable) headers: ['Name', 'City', 'County', 'Founded', 'Carnegie Classification[a]', 'Profit[b]', 'Enrollment (fall 2024)', 'Affiliation']
Processing Table 4 (wikitable sortable) headers: ['School', 'Location', 'Founded', 'Closed']

Final count: Found 0 universities.
First 5 extracted universities:


**Reasoning**:
The previous attempts failed because the relevant table containing university websites was not correctly identified or parsed. I need to modify the code to iterate through *all* tables on the Wikipedia page, identify tables that explicitly contain a 'Website' column in their headers, and then extract the university name and its corresponding URL from those specific tables.



In [ ]:
import requests
from bs4 import BeautifulSoup

wikipedia_url = "https://en.wikipedia.org/wiki/List_of_colleges_and_universities_in_California"

# Add a User-Agent header to the request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(wikipedia_url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

california_universities = {}

# Find all tables on the page
all_tables = soup.find_all('table')

print(f"Found {len(all_tables)} tables in total. Processing them to find relevant data.")

for table_idx, table in enumerate(all_tables):
    rows = table.find_all('tr')

    if not rows or len(rows) < 2:
        # print(f"Skipping table {table_idx}: not enough rows.") # Too verbose
        continue

    # Extract headers from the first row of the table
    headers_raw = [th.get_text(strip=True) for th in rows[0].find_all('th')]
    headers_lower = [h.lower() for h in headers_raw]

    university_name_col_index = -1
    website_col_index = -1

    # Find the column index for the university name
    for idx, header in enumerate(headers_lower):
        if "institution" in header or "name" in header or "university" in header or "school" in header:
            university_name_col_index = idx
            break

    # Find the column index for the website URL
    for idx, header in enumerate(headers_lower):
        if "website" in header:
            website_col_index = idx
            break

    # If both required columns are found, process this table
    if university_name_col_index != -1 and website_col_index != -1:
        print(f"\nProcessing Table {table_idx} (Classes: {table.get('class')}) with headers: {headers_raw}")
        print(f"  University name column index: {university_name_col_index}")
        print(f"  Website column index: {website_col_index}")

        # Iterate over data rows (skip header row)
        for row_num, row in enumerate(rows[1:]):
            cols = row.find_all('td')

            # Ensure there are enough columns to access the required indices
            if len(cols) > max(university_name_col_index, website_col_index):
                university_name_tag = cols[university_name_col_index].find('a')

                if university_name_tag and university_name_tag.text:
                    university_name = university_name_tag.text.strip()

                    website_link_tag = cols[website_col_index].find('a', href=True)

                    # Ensure it's a valid external URL (starts with http/https and not wikipedia)
                    if website_link_tag and website_link_tag['href'].startswith(('http://', 'https://')) and 'wikipedia.org' not in website_link_tag['href']:
                        main_url = website_link_tag['href']
                        california_universities[university_name] = {
                            'main_url': main_url,
                            'academic_programs_url': None,
                            'majors_url': None,
                            'course_catalog_url': None
                        }
                        # print(f"    Extracted: {university_name} - {main_url}") # Too verbose

print(f"\nFinal count: Found {len(california_universities)} universities.")
print("First 5 extracted universities:")
for i, (name, data) in enumerate(california_universities.items()):
    if i >= 5: break
    print(f"- {name}: {data['main_url']}")


Found 11 tables in total. Processing them to find relevant data.

Final count: Found 0 universities.
First 5 extracted universities:


In [ ]:
import requests
from bs4 import BeautifulSoup

wikipedia_url = "https://en.wikipedia.org/wiki/List_of_colleges_and_universities_in_California"

# Add a User-Agent header to the request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(wikipedia_url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

california_universities = {}

# Focus on 'wikitable sortable' tables, as they contain structured lists of universities
sortable_tables = soup.find_all('table', class_='wikitable sortable')

print(f"Found {len(sortable_tables)} tables with class 'wikitable sortable'.")

for table_idx, table in enumerate(sortable_tables):
    rows = table.find_all('tr')

    # Skip if no rows or only header row
    if not rows or len(rows) < 2:
        continue

    # Extract headers from the first row to determine column indices
    headers_raw = [th.get_text(strip=True) for th in rows[0].find_all('th')]
    headers_lower = [h.lower() for h in headers_raw]

    university_name_col_index = -1

    # Find the column index for the university name
    for idx, header in enumerate(headers_lower):
        if "institution" in header or "name" in header or "university" in header or "school" in header:
            university_name_col_index = idx
            break

    # If no university name column is found, skip this table
    if university_name_col_index == -1:
        continue

    # Iterate over data rows (skip header row)
    for row in rows[1:]:
        cols = row.find_all('td')

        # Ensure there are enough columns to access the university name index
        if len(cols) > university_name_col_index:
            university_name_tag = cols[university_name_col_index].find('a')

            if university_name_tag and university_name_tag.text:
                university_name = university_name_tag.text.strip()
                # Extract the Wikipedia page URL for the university
                wiki_page_url = None
                if university_name_tag.has_attr('href'):
                    relative_path = university_name_tag['href']
                    if relative_path.startswith('/wiki/'):
                        wiki_page_url = f"https://en.wikipedia.org{relative_path}"

                if wiki_page_url:
                    california_universities[university_name] = {
                        'main_url': wiki_page_url, # Store Wikipedia page URL for now
                        'academic_programs_url': None,
                        'majors_url': None,
                        'course_catalog_url': None
                    }

print(f"\nFinal count: Found {len(california_universities)} universities.")
print("First 5 extracted universities:")
for i, (name, data) in enumerate(california_universities.items()):
    if i >= 5: break
    print(f"- {name}: {data['main_url']}")

Found 5 tables with class 'wikitable sortable'.

Final count: Found 190 universities.
First 5 extracted universities:
- University of California, Berkeley: https://en.wikipedia.org/wiki/University_of_California,_Berkeley
- University of California, Davis: https://en.wikipedia.org/wiki/University_of_California,_Davis
- University of California, Irvine: https://en.wikipedia.org/wiki/University_of_California,_Irvine
- University of California, Los Angeles: https://en.wikipedia.org/wiki/University_of_California,_Los_Angeles
- University of California, Merced: https://en.wikipedia.org/wiki/University_of_California,_Merced


### Step 2: Extract Official Website URLs from Wikipedia Pages

Having obtained the Wikipedia page URLs for each university, the next task is to visit each of these pages. On each university's Wikipedia page, we will look for external links that typically point to the institution's official website. These are often found in the 'External links' section, infobox, or as a prominent link near the top of the page. We will update the `main_url` for each university in our `california_universities` dictionary with its official website URL.

### Step 2: Extract Official Website URLs from Wikipedia Pages

Having obtained the Wikipedia page URLs for each university, the next task is to visit each of these pages. On each university's Wikipedia page, we will look for external links that typically point to the institution's official website. These are often found in the 'External links' section, infobox, or as a prominent link near the top of the page. We will update the `main_url` for each university in our `california_universities` dictionary with its official website URL.

## Extract Official Website URLs from Wikipedia Pages

### Subtask:
Visit each university's Wikipedia page (whose URLs were previously scraped) to find and extract the official website URL. Update the `main_url` entry in the `california_universities` dictionary with the found official website URL.


**Reasoning**:
Now that the Wikipedia page URLs for California universities have been successfully extracted and stored, the next step is to visit each of these Wikipedia pages to find and extract the actual official university website URLs. This requires iterating through the `california_universities` dictionary, making a request to each Wikipedia URL, parsing the content, and then searching for external links that represent the university's official website.



In [ ]:
import requests
from bs4 import BeautifulSoup
import time

# User-Agent header to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# A dictionary to store universities with their official website URLs
official_california_universities = {}

print(f"Attempting to extract official websites for {len(california_universities)} universities...")

for i, (uni_name, uni_data) in enumerate(california_universities.items()):
    wiki_page_url = uni_data['main_url']
    official_website_found = None

    try:
        # Introduce a small delay to be polite to Wikipedia servers
        time.sleep(0.5)
        response = requests.get(wiki_page_url, headers=headers, timeout=10)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')

        # --- Strategy 1: Look in Infobox ---
        # Infoboxes often have a 'Website' or 'Official website' field.
        # The 'infobox' class is common for these tables.
        infobox = soup.find('table', class_='infobox')
        if infobox:
            for row in infobox.find_all('tr'):
                header = row.find('th')
                data_cell = row.find('td')
                if header and data_cell:
                    if 'website' in header.get_text(strip=True).lower() or 'official website' in header.get_text(strip=True).lower():
                        link = data_cell.find('a', href=True)
                        if link and link['href'].startswith(('http://', 'https://')) and 'wikipedia.org' not in link['href']:
                            official_website_found = link['href']
                            break
            if official_website_found: # If found in infobox, prioritize it
                # print(f"  Found official website for {uni_name} in infobox: {official_website_found}")
                pass

        # --- Strategy 2: Look for 'Official website' link text, typically in 'External links' section ---
        if not official_website_found:
            for link in soup.find_all('a', href=True):
                link_text = link.get_text(strip=True).lower()
                # Common link texts for official websites
                if 'official website' in link_text or 'homepage' in link_text or 'website' in link_text:
                    href = link['href']
                    if href.startswith(('http://', 'https://')) and 'wikipedia.org' not in href:
                        official_website_found = href
                        # print(f"  Found official website for {uni_name} in general links: {official_website_found}")
                        break

        # --- Strategy 3: Look for external links in the general content, particularly in the lead section ---
        # Sometimes the official website is linked early in the article body without explicit 'website' text.
        if not official_website_found:
            # Try to find common external link patterns in the initial paragraphs
            for p in soup.find_all('p', limit=3): # Check first few paragraphs
                for link in p.find_all('a', class_='external text', href=True):
                    href = link['href']
                    if href.startswith(('http://', 'https://')) and 'wikipedia.org' not in href:
                        # Heuristic: Filter out very generic links like .edu domains that aren't the primary site
                        # This might be too aggressive, but a starting point
                        if uni_name.lower().replace(' ', '') in href.lower().replace(' ', '') or \
                           uni_name.split(',')[0].lower().replace(' ', '') in href.lower().replace(' ', '') or \
                           uni_name.split(' ')[0].lower().replace(' ', '') in href.lower().replace(' ', ''):
                            official_website_found = href
                            # print(f"  Found official website for {uni_name} in lead section: {official_website_found}")
                            break
                if official_website_found: break

    except requests.exceptions.RequestException as e:
        print(f"  Error fetching {wiki_page_url}: {e}")
        # If an error occurs, keep the Wikipedia URL for now or set to None
        official_website_found = None
    except Exception as e:
        print(f"  An unexpected error occurred for {uni_name} ({wiki_page_url}): {e}")
        official_website_found = None

    # Update the dictionary. If no official website found, keep the Wikipedia URL.
    official_california_universities[uni_name] = {
        'main_url': official_website_found if official_website_found else wiki_page_url,
        'academic_programs_url': None,
        'majors_url': None,
        'course_catalog_url': None
    }

print(f"\nSuccessfully processed {len(official_california_universities)} universities.")
print("First 5 universities with updated main_url:")
for i, (name, data) in enumerate(official_california_universities.items()):
    if i >= 5: break
    print(f"- {name}: {data['main_url']}")

# Replace the original dictionary with the updated one
california_universities = official_california_universities

Attempting to extract official websites for 190 universities...

Successfully processed 190 universities.
First 5 universities with updated main_url:
- University of California, Berkeley: https://berkeley.edu
- University of California, Davis: https://ucdavis.edu
- University of California, Irvine: https://uci.edu
- University of California, Los Angeles: https://www.ucla.edu/
- University of California, Merced: https://www.ucmerced.edu


## Enhance Web Scraper for Detailed Information

### Subtask:
Modify the existing `scrape_page` function or create a new one to not only extract text from general pages but also to identify and scrape specific sections related to majors, departments, and course descriptions from the identified university URLs. This might involve more targeted HTML parsing.


**Reasoning**:
I need to define a new function `scrape_university_details` that will take a university's main URL, fetch its content, and then attempt to find links for academic programs, majors, and course catalogs using keyword-based heuristics. This function will return a dictionary of these specialized URLs.



In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse

def scrape_university_details(university_url):
    """
    Scrapes a university's main page to find links for academic programs, majors, and course catalogs.
    """
    details = {
        'academic_programs_url': None,
        'majors_url': None,
        'course_catalog_url': None
    }

    if not university_url:
        # print(f"Skipping {university_url}: URL is empty.") # Debugging
        return details

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(university_url, headers=headers, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Keywords to search for in link text and hrefs
        academic_keywords = ['academics', 'programs', 'study', 'education', 'degrees', 'departments']
        majors_keywords = ['majors', 'degrees', 'undergraduate', 'graduate', 'programs of study', 'fields of study']
        catalog_keywords = ['catalog', 'courses', 'curriculum', 'course catalog', 'bulletin', 'syllabi']

        # To store potential URLs with their scores (how relevant they seem)
        # Store as (score, url_length, url) to prioritize by score, then by shorter URL
        best_academic_link = (0, 0, None)
        best_majors_link = (0, 0, None)
        best_catalog_link = (0, 0, None)

        # Iterate through all links on the page
        for link in soup.find_all('a', href=True):
            href = link['href']
            link_text = link.get_text(strip=True).lower()

            # Resolve relative URLs to absolute URLs
            absolute_href = urljoin(university_url, href)

            # Skip external links or non-relevant links (e.g., social media, internal anchors)
            # Ensure the link is within the same domain or a subdomain
            parsed_current_url = urlparse(university_url)
            parsed_absolute_href = urlparse(absolute_href)
            if parsed_absolute_href.netloc != parsed_current_url.netloc and \
               not parsed_absolute_href.netloc.endswith('.' + parsed_current_url.netloc) or '#' in href or not absolute_href.startswith(('http://', 'https://')):
                continue

            score_academic = 0
            score_majors = 0
            score_catalog = 0

            # Score based on keywords in link text and href
            for keyword in academic_keywords:
                if keyword in link_text:
                    score_academic += 2
                if keyword in parsed_absolute_href.path.lower(): # Search path for keywords
                    score_academic += 1

            for keyword in majors_keywords:
                if keyword in link_text:
                    score_majors += 2
                if keyword in parsed_absolute_href.path.lower():
                    score_majors += 1

            for keyword in catalog_keywords:
                if keyword in link_text:
                    score_catalog += 2
                if keyword in parsed_absolute_href.path.lower():
                    score_catalog += 1

            # Update best links found
            current_academic_score = (score_academic, -len(absolute_href), absolute_href) # Negative length for shorter URLs
            if current_academic_score[0] > best_academic_link[0] or \
               (current_academic_score[0] == best_academic_link[0] and current_academic_score[1] > best_academic_link[1]):
                best_academic_link = current_academic_score

            current_majors_score = (score_majors, -len(absolute_href), absolute_href)
            if current_majors_score[0] > best_majors_link[0] or \
               (current_majors_score[0] == best_majors_link[0] and current_majors_score[1] > best_majors_link[1]):
                best_majors_link = current_majors_score

            current_catalog_score = (score_catalog, -len(absolute_href), absolute_href)
            if current_catalog_score[0] > best_catalog_link[0] or \
               (current_catalog_score[0] == best_catalog_link[0] and current_catalog_score[1] > best_catalog_link[1]):
                best_catalog_link = current_catalog_score

        if best_academic_link[0] > 0: # Only assign if a relevant link was found
            details['academic_programs_url'] = best_academic_link[2]
        if best_majors_link[0] > 0:
            details['majors_url'] = best_majors_link[2]
        if best_catalog_link[0] > 0:
            details['course_catalog_url'] = best_catalog_link[2]

    except requests.exceptions.HTTPError as e:
        # print(f"  HTTP Error for {university_url}: {e}") # Debugging
        pass
    except requests.exceptions.ConnectionError as e:
        # print(f"  Connection Error for {university_url}: {e}") # Debugging
        pass
    except requests.exceptions.Timeout as e:
        # print(f"  Timeout Error for {university_url}: {e}") # Debugging
        pass
    except requests.exceptions.RequestException as e:
        # print(f"  Request Error for {university_url}: {e}") # Debugging
        pass
    except Exception as e:
        # print(f"  An unexpected error occurred for {university_url}: {e}") # Debugging
        pass

    return details

# Iterate through the california_universities and scrape details
print(f"Starting detailed scraping for {len(california_universities)} universities...")
updated_universities_with_details = {}
for i, (uni_name, uni_data) in enumerate(california_universities.items()):
    # print(f"Processing {i+1}/{len(california_universities)}: {uni_name}") # Debugging
    main_url = uni_data['main_url']
    scraped_details = scrape_university_details(main_url)

    # Update the university data with scraped details
    updated_universities_with_details[uni_name] = {
        'main_url': main_url,
        'academic_programs_url': scraped_details['academic_programs_url'],
        'majors_url': scraped_details['majors_url'],
        'course_catalog_url': scraped_details['course_catalog_url']
    }
    time.sleep(0.1) # Be polite to university servers

california_universities = updated_universities_with_details

print(f"\nFinished detailed scraping. Updated {len(california_universities)} universities.")
print("First 5 universities with extracted details:")
for i, (name, data) in enumerate(california_universities.items()):
    if i >= 5: break
    print(f"- {name}:")
    print(f"  Main URL: {data['main_url']}")
    print(f"  Academic Programs URL: {data['academic_programs_url']}")
    print(f"  Majors URL: {data['majors_url']}")
    print(f"  Course Catalog URL: {data['course_catalog_url']}")


Starting detailed scraping for 190 universities...

Finished detailed scraping. Updated 190 universities.
First 5 universities with extracted details:
- University of California, Berkeley:
  Main URL: https://berkeley.edu
  Academic Programs URL: None
  Majors URL: None
  Course Catalog URL: None
- University of California, Davis:
  Main URL: https://ucdavis.edu
  Academic Programs URL: https://ucdavis.edu/academics/study-abroad
  Majors URL: https://ucdavis.edu/research/undergraduate
  Course Catalog URL: https://catalog.ucdavis.edu/
- University of California, Irvine:
  Main URL: https://uci.edu
  Academic Programs URL: https://uci.edu//community/education-programs.php
  Majors URL: https://uci.edu//academics/undergraduate.php
  Course Catalog URL: None
- University of California, Los Angeles:
  Main URL: https://www.ucla.edu/
  Academic Programs URL: https://www.ucla.edu/academics/programs-and-majors
  Majors URL: https://www.ucla.edu/academics/undergraduate-education
  Course Catal

## Process and Chunk Scraped Data

### Subtask:
Apply the existing `chunk_text` function to the newly scraped university data, ensuring chunks are semantically coherent and include metadata such as university name, major area, or course title.


**Reasoning**:
The subtask requires iterating through the scraped university URLs (academic programs, majors, course catalogs), fetching their content, chunking the text using the `chunk_text` function, and associating each chunk with relevant metadata. I need to create a loop that iterates through the `california_universities` dictionary and applies the scraping, chunking, and metadata tagging logic for each applicable URL.



In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse

# Reuse the chunk_text function defined earlier
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

# Reuse the scrape_page function, or adapt it to be more robust
def get_text_from_url(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')
        paragraphs = [p.get_text() for p in soup.find_all("p")]
        # Also extract text from headings for better context
        headings = [h.get_text() for h in soup.find_all(['h1', 'h2', 'h3', 'h4'])]
        return " ".join(headings + paragraphs)
    except requests.exceptions.RequestException as e:
        # print(f"  Error fetching {url}: {e}") # Debugging
        return None
    except Exception as e:
        # print(f"  An unexpected error occurred for {url}: {e}") # Debugging
        return None

all_university_chunks = []
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f"Starting to process and chunk data for {len(california_universities)} universities...")

for uni_name, uni_data in california_universities.items():
    for url_type, url in uni_data.items():
        if url_type == 'main_url': # Skip main_url for detailed content scraping, unless specific context is needed
            continue

        if url:
            # print(f"  Scraping {url_type} for {uni_name} from: {url}") # Debugging
            text_content = get_text_from_url(url, headers)

            if text_content:
                chunks = chunk_text(text_content, chunk_size=500) # Use the previously defined chunk_text
                for chunk_text_data in chunks:
                    all_university_chunks.append({
                        'text': chunk_text_data,
                        'university_name': uni_name,
                        'url_type': url_type.replace('_url', ''), # e.g., 'academic_programs'
                        'source_url': url
                    })
            time.sleep(0.1) # Be polite

print(f"\nFinished processing. Total chunks created: {len(all_university_chunks)}")

print("\nFirst 5 example chunks with metadata:")
for i, chunk in enumerate(all_university_chunks):
    if i >= 5: break
    print(f"-- Chunk {i+1} --")
    print(f"  University: {chunk['university_name']}")
    print(f"  Type: {chunk['url_type']}")
    print(f"  Source URL: {chunk['source_url']}")
    print(f"  Text (first 200 chars): {chunk['text'][:200]}...")
    print("\n")

Starting to process and chunk data for 190 universities...

Finished processing. Total chunks created: 809

First 5 example chunks with metadata:
-- Chunk 1 --
  University: University of California, Davis
  Type: academic_programs
  Source URL: https://ucdavis.edu/academics/study-abroad
  Text (first 200 chars): UC Davis Main navigation (extended config) A global adventure calls Gaining global skills while making a difference Additional Global Learning Hub resources What Can I Do With My Major? Blog Sub Main ...


-- Chunk 2 --
  University: University of California, Davis
  Type: majors
  Source URL: https://ucdavis.edu/research/undergraduate
  Text (first 200 chars): UC Davis Main navigation (extended config) Undergraduate Research Students connect to hands-on learning Getting started Fund your research Find an internship Define your college career with a capstone...


-- Chunk 3 --
  University: University of California, Davis
  Type: course_catalog
  Source URL: https://catalog.uc

## Generate Embeddings

### Subtask:
Use an embedding model to convert the processed text chunks into vector embeddings.


**Reasoning**:
I will instantiate the LiteLlm embedding model and then iterate through the `all_university_chunks` list, generating an embedding for the 'text' content of each chunk and storing it back in the chunk dictionary under the 'embedding' key.



In [ ]:
import litellm

# The embedding model name
EMBEDDING_MODEL_NAME = "text-embedding-004"

print(f"Generating embeddings for {len(all_university_chunks)} chunks using litellm.embedding...")

# Iterate through each chunk and generate its embedding
for i, chunk in enumerate(all_university_chunks):
    try:
        # Use litellm.embedding directly
        response = litellm.embedding(model=EMBEDDING_MODEL_NAME, input=[chunk['text']])

        if response and hasattr(response, 'data') and response.data and hasattr(response.data[0], 'embedding'):
            chunk['embedding'] = response.data[0].embedding # Store the embedding values
        else:
            chunk['embedding'] = [] # Assign an empty list if embedding not found
            print(f"Warning: Embedding not generated for chunk {i+1}. Response was: {response}")
    except Exception as e:
        chunk['embedding'] = [] # Assign an empty list if an error occurs
        print(f"Error generating embedding for chunk {i+1}: {e}")

print(f"Finished generating embeddings. First 5 chunks with embeddings:\n")
for i, chunk in enumerate(all_university_chunks):
    if i >= 5: break
    print(f"-- Chunk {i+1} --")
    print(f"  University: {chunk['university_name']}")
    print(f"  Type: {chunk['url_type']}")
    print(f"  Embedding length: {len(chunk['embedding'])}")
    print(f"  Text (first 50 chars): {chunk['text'][:50]}...")
    print("\n")